# ⚡ AI B-Roll Autopilot — Colab Free GPU Backend
Run your heavy Whisper transcription, Remotion rendering, and B-roll pipeline with a free **T4 / A100 GPU**.

### ⚙️ Make sure GPU is selected:
`Runtime` → `Change runtime type` → `T4 GPU` → `Save`

In [ ]:
# @title 🚀 1. Install Dependencies & Clone Repo
import os

print("Checking GPU...")
!nvidia-smi

print("\nCloning latest stockpile repo...")
!rm -rf /content/stockpile
!git clone -b sorry-i-fucked-up https://github.com/Sh1sh1R017/stockpile.git /content/stockpile
%cd /content/stockpile

print("\nInstalling ffmpeg & packages...")
!apt-get update -qq && apt-get install -y ffmpeg -qq
!pip install -q -r requirements.txt
!pip install -q pycloudflared uvicorn[standard] python-multipart
print("✅ Setup complete!")

In [ ]:
# @title 🔑 2. Configure API Keys (Optional)
# @markdown Enter your API keys (or leave blank if testing with fallbacks)
GEMINI_API_KEY = "" # @param {type:"string"}
PEXELS_API_KEY = "" # @param {type:"string"}
GIPHY_API_KEY = "" # @param {type:"string"}

with open("/content/stockpile/.env", "w") as f:
    if GEMINI_API_KEY.strip():
        f.write(f"GEMINI_API_KEY={GEMINI_API_KEY.strip()}\n")
    if PEXELS_API_KEY.strip():
        f.write(f"PEXELS_API_KEY={PEXELS_API_KEY.strip()}\n")
    if GIPHY_API_KEY.strip():
        f.write(f"GIPHY_API_KEY={GIPHY_API_KEY.strip()}\n")
    f.write("LOCAL_INPUT_FOLDER=input\nLOCAL_OUTPUT_FOLDER=output\nWHISPER_MODEL=small\n")

print("✅ Configuration saved to .env!")

In [ ]:
# @title 🌐 3. Launch GPU Backend + Cloudflare Tunnel
import threading
import time
import os
from pycloudflared import try_cloudflare

%cd /content/stockpile

# Start FastAPI
def run_backend():
    os.system("python -m ai_broll_autopilot.cli serve --host 0.0.0.0 --port 8000")

backend_thread = threading.Thread(target=run_backend, daemon=True)
backend_thread.start()
time.sleep(4)

# Tunnel port 8000 to public HTTPS
tunnel = try_cloudflare(port=8000)
tunnel_url = tunnel.tunnel.url

print("\n" + "="*70)
print("🚀 YOUR GOOGLE COLAB GPU BACKEND IS READY!")
print(f"🔗 PUBLIC API URL: {tunnel_url}")
print("="*70)
print("\n📋 TO CONNECT YOUR LOCAL WEB STUDIO:")
print(f"1. In your project directory, set BACKEND_API_URL in 'web/.env.local':")
print(f"   BACKEND_API_URL={tunnel_url}")
print("2. Or start your Next.js dev server with:")
print(f"   $env:BACKEND_API_URL='{tunnel_url}'; npm run dev")
print("="*70 + "\n")

while True:
    time.sleep(60)